> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.
>
> The notebook also uses `scikit-learn`, `xgboost`, which are not shipped with pdstools — install them separately.


In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately

import plotly.io as pio

pio.renderers.default = "notebook_connected"

# HDS (Historical Data Set) Analysis and Finding new Features

This notebook demonstrates how to work with a sample HDS dataset, explore its structure, and build a simple XGBoost model to show feature importance.

It then shows how to combine this data with a sample dataset from the Data Lake and perform an analysis of which features could be considered to pull in to Pega to improve the models.

In [2]:
import polars as pl
import polars.selectors as cs
import plotly.express as px
import requests, zipfile, io
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from pdstools.utils import cdh_utils

## Data Read

Read the HDS data. Depending on how the data was extracted and stored, you may need different reading methods using
zipfile and/or Polars.

We're also casting the data to the appropriate types and dropping some features that shouldnt be used for models. 

In [3]:
if Path("../../data/hds.zip").exists():
    archive = zipfile.ZipFile("../../data/hds.zip", "r")
else:
    # Download the dataset if it does not exist locally
    archive = zipfile.ZipFile(io.BytesIO(requests.get('https://github.com/pegasystems/pega-datascientist-tools/raw/master/data/hds.zip').content))
hds_data = (
    pl.concat([pl.read_ndjson(archive.open(f)) for f in archive.namelist()])
    .rename({"Customer_C_CIFNBR": "Customer_ID"})
    .with_columns(
        cdh_utils.parse_pega_date_time_formats("Decision_DecisionTime"),
        cdh_utils.parse_pega_date_time_formats("Decision_OutcomeTime"),
        cs.ends_with("_DaysSince", "_pyHistoricalOutcomeCount").cast(pl.Float64),
        pl.col(
            [
                "Customer_NetWealth",
                "Customer_CreditScore",
                "Customer_CLV_VALUE",
                "Customer_RelationshipStartDate",
                "Customer_Date_of_Birth",
                "Customer_NoOfDependents"
            ]
        ).cast(pl.Float64),
        cs.starts_with("Param_ExtGroup").cast(pl.Float64),
    )
    .drop(["Customer_Gender", "Customer_Prefix"])
    .sort(["Customer_ID", "Decision_DecisionTime"])
)
hds_data.describe()

statistic,Decision_SubjectID,IH_Email_Outbound_Loyal_pyHistoricalOutcomeCount,Customer_ReviewDate,Customer_IsCustomerActive,Customer_Date_of_Birth,Customer_pyRegion,Customer_pyCountry,IH_SMS_Outbound_Churned_pxLastGroupID,IH_Email_Outbound_Accepted_pxLastOutcomeTime_DaysSince,IH_SMS_Outbound_Churned_pyHistoricalOutcomeCount,IH_Web_Inbound_Accepted_pxLastGroupID,IH_SMS_Outbound_Churned_pxLastOutcomeTime_DaysSince,positiveSampling,Customer_HealthMatter,IH_SMS_Outbound_Accepted_pxLastGroupID,Customer_CLV,IH_Web_Inbound_Loyal_pyHistoricalOutcomeCount,Decision_Rank,Customer_RelationshipStartDate,IH_Email_Outbound_Rejected_pxLastOutcomeTime_DaysSince,IH_Web_Inbound_Accepted_pyHistoricalOutcomeCount,IH_Web_Inbound_Accepted_pxLastOutcomeTime_DaysSince,rulesetVersion,Decision_DecisionTime,Customer_NoOfDependents,Customer_NaturalDisaster,Customer_NetWealth,Decision_OutcomeTime,Context_Name,Customer_RiskScore,IH_SMS_Outbound_Rejected_pxLastGroupID,Param_ExtGroupAutoloans,Customer_BalanceTransaction,Customer_Bankruptcy,IH_SMS_Outbound_Rejected_pxLastOutcomeTime_DaysSince,IH_SMS_Outbound_Rejected_pyHistoricalOutcomeCount,…,Customer_AnnualIncome,Customer_NextReviewDate,Context_Channel,Param_ExtGroupWealthoffers,IH_SMS_Outbound_Loyal_pxLastOutcomeTime_DaysSince,Customer_MaritalStatus,IH_Email_Outbound_Churned_pxLastOutcomeTime_DaysSince,IH_Email_Outbound_Rejected_pyHistoricalOutcomeCount,dataCenter,IH_SMS_Outbound_Loyal_pyHistoricalOutcomeCount,IH_Web_Inbound_Rejected_pyHistoricalOutcomeCount,IH_Email_Outbound_Loyal_pxLastGroupID,Customer_TotalAssets,IH_SMS_Outbound_Loyal_pxLastGroupID,IH_Email_Outbound_Loyal_pxLastOutcomeTime_DaysSince,Customer_City,id,Customer_CreditScore,IH_SMS_Outbound_Accepted_pxLastOutcomeTime_DaysSince,Customer_WinScore,IH_Web_Inbound_Loyal_pxLastOutcomeTime_DaysSince,Param_ExtGroupCreditcards,Decision_InteractionID,Customer_OrganizationLabel,IH_Email_Outbound_Rejected_pxLastGroupID,Customer_Incarceration,IH_Email_Outbound_Accepted_pxLastGroupID,IH_Web_Inbound_Churned_pxLastGroupID,Customer_ResidentialStatus,IH_Email_Outbound_Churned_pyHistoricalOutcomeCount,IH_Email_Outbound_Churned_pxLastGroupID,IH_SMS_Outbound_Accepted_pyHistoricalOutcomeCount,Context_Issue,Context_Direction,Customer_Age,Customer_CLV_VALUE,Customer_MilitaryService
str,str,f64,str,str,f64,str,str,str,f64,f64,str,f64,str,str,str,str,f64,str,f64,f64,f64,f64,str,str,f64,str,f64,str,str,str,str,f64,str,str,f64,f64,…,str,str,str,f64,f64,str,f64,f64,str,f64,f64,str,str,str,f64,str,str,f64,f64,str,f64,f64,str,str,str,str,str,str,str,f64,str,f64,str,str,str,f64,str
"""count""","""12062""",2432.0,"""12062""","""12062""",12062.0,"""12062""","""12062""","""828""",8156.0,828.0,"""11971""",828.0,"""12062""","""12062""","""11826""","""12062""",6076.0,"""12062""",12062.0,12062.0,11971.0,11971.0,"""12062""","""12062""",12062.0,"""12062""",12062.0,"""12062""","""12062""","""12062""","""12062""",12062.0,"""12062""","""12062""",12062.0,12062.0,…,"""12062""","""12062""","""12062""",12062.0,1169.0,"""12062""",2259.0,12062.0,"""12062""",1169.0,12062.0,"""2432""","""12062""","""1169""",2432.0,"""12062""","""12062""",12062.0,11826.0,"""12062""",6076.0,12062.0,"""12062""","""12062""","""12062""","""12062""","""8156""","""5911""","""12062""",2259.0,"""2259""",11826.0,"""12062""","""12062""","""11947""",12062.0,"""12062"""
"""null_count""","""0""",9630.0,"""0""","""0""",0.0,"""0""","""0""","""11234""",3906.0,11234.0,"""91""",11234.0,"""0""","""0""","""236""","""0""",5986.0,"""0""",0.0,0.0,91.0,91.0,"""0""","""0""",0.0,"""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0,…,"""0""","""0""","""0""",0.0,10893.0,"""0""",9803.0,0.0,"""0""",10893.0,0.0,"""9630""","""0""","""10893""",9630.0,"""0""","""0""",0.0,236.0,"""0""",5986.0,0.0,"""0""","""0""","""0""","""0""","""3906""","""6151""","""0""",9803.0,"""9803""",236.0,"""0""","""0""","""115""",0.0,"""0"""
"""mean""",null,1.271793,null,null,18775.43305,null,null,null,51.498683,1.07971,null,61.296615,null,null,null,

## Available Fields in the HDS dataset

The HDS data contains all the payload sent to the ADM models (over a period of time) plus the outcomes (Accepted/Declined/Clicked etc). There are a few categories of fields, that can be identified by their prefix:

* "Customer" fields, representing the fields/predictors configured in ADM
* "Context" fields, these are Channel/Direction/Issue/Group/Name, the usual "context identifiers" of the ADM models
* "IH" fields, these are Pega-generated fields derived from Interaction History
* Optional "Param" fields, also user defined fields/predictors, but configured in the strategies, rather than defined in the ADM model configuration

Meta information about the decisions is in Decision and internal fields, containing info about the time of decision, the sample size etc. These are not used in the models.



In [4]:
hds_data_dictionary = (
    pl.DataFrame(
           {"Field" : hds_data.schema.names(),
           "Numeric" : [x.is_numeric() for x in hds_data.schema.dtypes()],}
    )
    .with_columns(
        Category=pl.when(pl.col("Field").str.contains("_", literal=True))
        .then(pl.col("Field").str.replace(r"([^_]+)_.*", "${1}"))
        .otherwise(pl.lit("Internal"))
    )
    .sort("Category")
)
hds_data_dictionary.to_pandas().style.hide()


Field,Numeric,Category
Context_Name,False,Context
Context_Group,False,Context
Context_Channel,False,Context
Context_Issue,False,Context
Context_Direction,False,Context
Customer_ReviewDate,False,Customer
Customer_IsCustomerActive,False,Customer
Customer_Date_of_Birth,True,Customer
Customer_pyRegion,False,Customer
Customer_pyCountry,False,Customer


In [5]:
category_counts = (
    hds_data_dictionary
    .group_by("Category", "Numeric")
    .agg(Count=pl.len())
    .sort("Category")
)
fig = px.bar(
    category_counts, #.to_dict(as_series=False),
    y="Category",
    x="Count",
    title="Number of Fields by Category",
    color="Numeric",
    text="Count",
    orientation="h",
)

fig.update_layout(
    yaxis_title="Field Category",
    xaxis_title="Number of Fields",
)

fig.show()

## Create XGBoost model 

From this HDS data we create a simple XGBoost model. We create a simple model over all channels and all actions, not split up like ADM does. This could be changed, but the goal here is to get a list of the features ranked by importance, not to create a model that is better than ADM.

First, there is data prep to do one-hot encoding and target encoding.

In [6]:
def data_prep(data, data_dictionary):
    categorical_fields = (
        data_dictionary.filter(~pl.col("Numeric"))
        .filter((pl.col("Category") != "Internal") & (pl.col("Category") != "Decision"))
        .select("Field")
        .to_series()
        .to_list()
    )
    numerical_fields = (
        data_dictionary.filter(pl.col("Numeric"))
        .filter((pl.col("Category") != "Internal") & (pl.col("Category") != "Decision"))
        .select("Field")
        .to_series()
        .to_list()
    )

    print(f"Categorical fields: {categorical_fields}")
    print(f"Numerical fields: {numerical_fields}")

    # Simple encoding for categorical features
    for column in categorical_fields:
        if column != "Decision_Outcome":
            # Handle missing values
            data = data.with_columns(
                pl.col(column).fill_null("missing").alias(column)
            )

            # Create a simple label encoder
            le = LabelEncoder()
            encoded = le.fit_transform(data[column].to_list())
            data = data.with_columns(
                pl.Series(name=column + "_encoded", values=encoded)
            )

    # Encode target variable
    le_target = LabelEncoder()
    encoded_target = le_target.fit_transform(data["Decision_Outcome"].to_list())
    data = data.with_columns(
        pl.Series(name="target", values=encoded_target)
    )

    # Show target encoding
    target_mapping = dict(zip(le_target.classes_, range(len(le_target.classes_))))
    print(f"\nTarget encoding: {target_mapping}")

    # Select features and target
    feature_cols = [
        col for col in data.columns if col.endswith("_encoded")
    ] + numerical_fields

    X = data[feature_cols]
    y = data["target"]

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"\nTraining set size: {X_train.shape[0]} samples")
    print(f"Test set size: {X_test.shape[0]} samples")

    return X_train, X_test, y_train, y_test, le_target, feature_cols


X_train, X_test, y_train, y_test, target_encoder, feature_cols = data_prep(hds_data, hds_data_dictionary)

Categorical fields: ['Context_Name', 'Context_Group', 'Context_Channel', 'Context_Issue', 'Context_Direction', 'Customer_ReviewDate', 'Customer_IsCustomerActive', 'Customer_pyRegion', 'Customer_pyCountry', 'Customer_HealthMatter', 'Customer_CLV', 'Customer_NaturalDisaster', 'Customer_RiskScore', 'Customer_BalanceTransaction', 'Customer_Bankruptcy', 'Customer_RiskCode', 'Customer_State', 'Customer_Deceased', 'Customer_TotalLiabilities', 'Customer_LastReviewedDate', 'Customer_InCollections', 'Customer_ID', 'Customer_BusinessSegment', 'Customer_AnnualIncome', 'Customer_NextReviewDate', 'Customer_MaritalStatus', 'Customer_TotalAssets', 'Customer_City', 'Customer_WinScore', 'Customer_OrganizationLabel', 'Customer_Incarceration', 'Customer_ResidentialStatus', 'Customer_Age', 'Customer_MilitaryService', 'IH_SMS_Outbound_Churned_pxLastGroupID', 'IH_Web_Inbound_Accepted_pxLastGroupID', 'IH_SMS_Outbound_Accepted_pxLastGroupID', 'IH_SMS_Outbound_Rejected_pxLastGroupID', 'IH_Web_Inbound_Rejected_p

In [7]:
from xgboost import XGBClassifier
def create_classifier(X_train, X_test, y_train, y_test, target_encoder):
    # Create and train the model
    xgb_model = XGBClassifier(random_state=42)
    xgb_model.fit(X_train, y_train)

    # Make predictions and evaluate
    y_pred = xgb_model.predict(X_test)
    print (f"Model AUC: {round(roc_auc_score(y_test,y_pred), 5)}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

    return xgb_model

classifier = create_classifier(X_train, X_test, y_train, y_test, target_encoder)

Model AUC: 0.56175

Classification Report:
              precision    recall  f1-score   support

    Accepted       0.30      0.15      0.20       178
    Rejected       0.93      0.97      0.95      2235

    accuracy                           0.91      2413
   macro avg       0.62      0.56      0.58      2413
weighted avg       0.89      0.91      0.90      2413



In [8]:
def plot_feature_imp(classifier, data, feature_cols):
    importances = classifier.feature_importances_

    # Create a DataFrame for feature importances
    feature_importance_df = (
        pl.DataFrame({"Feature": feature_cols, "Importance": importances.tolist()})
        .with_columns(
            Feature=pl.when(pl.col("Feature").str.ends_with("_encoded"))
            .then(pl.col("Feature").str.replace(r"_encoded$", ""))
            .otherwise(pl.col("Feature"))
        )
        .with_columns(
            Category=pl.when(pl.col("Feature").str.contains("_", literal=True))
            .then(pl.col("Feature").str.replace(r"([^_]+)_.*", "${1}"))
            .otherwise(pl.lit("Internal"))
        )
        .sort("Importance", descending=True)
    )

    # Get top N features by importance
    top_features_df = feature_importance_df.head(50)

    # Get the ordered list of feature names
    feature_order = top_features_df["Feature"].to_list()

    # Correlation test of new features
    features_encoded_name = [
        f"{feat}_encoded" if f"{feat}_encoded" in feature_cols else feat
        for feat in feature_order
    ]   

    similar_features = []
    n = len(features_encoded_name)

    for i in range(n):
        for j in range(i + 1, n):
            col1 = features_encoded_name[i]
            col2 = features_encoded_name[j]

            correlation = data.select(pl.corr(col1, col2)).item()
            if abs(correlation) >= 0.95:
                found = False

                for i, tup in enumerate(similar_features):
                    if col1 in tup:
                        similar_features[i] = tup + (col2,)
                        found = True
                        break
                    elif col2 in tup:
                        similar_features[i] = tup + (col1,)
                        found = True

                if not found:
                    similar_features.append((col1, col2))
    

    # Creating the group label
    group_mapping = {}
    for i, group in enumerate(similar_features, 1):
        for feature in group:
            group_mapping[feature] = f"Group {i}"

    top_features_df = top_features_df.with_columns(
        pl.col("Feature").map_elements(lambda f: group_mapping.get(f, ""), return_dtype=pl.Utf8).alias("GroupLabel")
    )

    # Plot feature importances
    fig = px.bar(
        top_features_df,
        x="Importance",
        y="Feature",
        orientation="h",
        title="Feature Importance",
        template="plotly_white",
        color="Category",
        text="GroupLabel",
        color_discrete_map={
            "Context": "orange",
            "IH": "green",
            "Customer": "blue",
            "Param": "lightblue",
            "DataLake": "red",
            "Internal": "gray",
            "Decision": "purple",
        },
    )

    fig.update_layout(
        xaxis_title="Importance",
        yaxis_title="Feature",
        yaxis=dict(
            categoryorder="array",
            categoryarray=feature_order,
            autorange="reversed",
            dtick=1,
        ),
    )

    fig.update_traces(
        textposition="outside",
        textfont=dict(color="black", size=12),
    )
    return fig

plot_feature_imp(classifier, X_train, feature_cols).update_layout(height=800).show()

Features that are strongly correlated are shown with an indication of the group (e.g. "Group 1"). Colors are used to differentiate the different sources of the features. In this demo data set, you see that Group and Channel are very important features, as is expected.

## Finding new Features from the Data Lake

Now, suppose you have external data from your data lake that you want to consider adding to Pega to improve the performance of your models.

If you have such data, you can merge it with the HDS data and run the model again to see how these features fare against what ADM already uses.

Such data is typically time-stamped, so we need to be careful to only pull in data from before the decisions were made. 

### Create (fake) External Data

We first create an example of external data. All features are captured over time there, so there is a feature name, a timestamp, and a value.

This code (and resulting data) are just an example. You can use any data you want, we just highlight the structure.


In [9]:
import random

random.seed(101)
datalake_fake_data = hds_data.with_columns(
    DataLake_BadFeature=pl.Series([random.random() for _ in range(hds_data.height)]),
    DataLake_GoodFeature=(pl.col("Decision_Outcome") == "Accepted") * 0.9
    + pl.Series([random.random() for _ in range(hds_data.height)]) * 0.1,
    DataLake_GoodFeatureCorrelated=(pl.col("Decision_Outcome") == "Accepted") * 0.8
    + pl.Series([random.random() for _ in range(hds_data.height)]) * 0.1
).select(
    [
        pl.col("Customer_ID"),
        pl.col("Decision_DecisionTime").dt.truncate("1d").alias("SnapshotTime"),
        pl.col("DataLake_BadFeature"),
        pl.col("DataLake_GoodFeature"),
        pl.col("DataLake_GoodFeatureCorrelated")
    ]
).group_by(
    ["Customer_ID", "SnapshotTime"]
).agg(
    cs.all().mean()
).sort(["Customer_ID", "SnapshotTime"])

datalake_fake_data.head()

Customer_ID,SnapshotTime,DataLake_BadFeature,DataLake_GoodFeature,DataLake_GoodFeatureCorrelated
str,datetime[ns],f64,f64,f64
"""Customer-1253""",2021-05-24 00:00:00,0.451482,0.168608,0.149609
"""Customer-1254""",2021-05-24 00:00:00,0.409358,0.170598,0.150844
"""Customer-1255""",2021-05-24 00:00:00,0.476272,0.084989,0.089284
"""Customer-1256""",2021-05-24 00:00:00,0.538506,0.054512,0.048855
"""Customer-1257""",2021-05-24 00:00:00,0.526969,0.061005,0.04727


Joining that data with the HDS data is straightforward: we match by customer ID and timestamp, but need to be careful to avoid leakage, so we only join in data for a particular customer from the data lake that is the latest snapshot before the timestamp of the HDS dataset. 

Polars provides a convenient way to do this with the join_asof function.

In [10]:
augmented_data = hds_data.join_asof(
    datalake_fake_data,
    left_on="Decision_DecisionTime",
    right_on="SnapshotTime",
    by="Customer_ID",
    check_sortedness=False,
)
augmented_data_dictionary = pl.concat(
    [
        hds_data_dictionary,
        pl.DataFrame(
            {
                "Field": ["DataLake_BadFeature", "DataLake_GoodFeature", "DataLake_GoodFeatureCorrelated"],
                "Numeric": [True, True, True],
                "Category": ["DataLake", "DataLake", "DataLake"],
            }
        ),
    ]
)

In [11]:
X_train, X_test, y_train, y_test, target_encoder, feature_cols = data_prep(augmented_data, augmented_data_dictionary)
classifier = create_classifier(X_train, X_test, y_train, y_test, target_encoder)

plot_feature_imp(classifier, X_train, feature_cols).update_layout(height=800).show()

Categorical fields: ['Context_Name', 'Context_Group', 'Context_Channel', 'Context_Issue', 'Context_Direction', 'Customer_ReviewDate', 'Customer_IsCustomerActive', 'Customer_pyRegion', 'Customer_pyCountry', 'Customer_HealthMatter', 'Customer_CLV', 'Customer_NaturalDisaster', 'Customer_RiskScore', 'Customer_BalanceTransaction', 'Customer_Bankruptcy', 'Customer_RiskCode', 'Customer_State', 'Customer_Deceased', 'Customer_TotalLiabilities', 'Customer_LastReviewedDate', 'Customer_InCollections', 'Customer_ID', 'Customer_BusinessSegment', 'Customer_AnnualIncome', 'Customer_NextReviewDate', 'Customer_MaritalStatus', 'Customer_TotalAssets', 'Customer_City', 'Customer_WinScore', 'Customer_OrganizationLabel', 'Customer_Incarceration', 'Customer_ResidentialStatus', 'Customer_Age', 'Customer_MilitaryService', 'IH_SMS_Outbound_Churned_pxLastGroupID', 'IH_Web_Inbound_Accepted_pxLastGroupID', 'IH_SMS_Outbound_Accepted_pxLastGroupID', 'IH_SMS_Outbound_Rejected_pxLastGroupID', 'IH_Web_Inbound_Rejected_p

Model AUC: 0.5567

Classification Report:
              precision    recall  f1-score   support

    Accepted       0.26      0.15      0.19       178
    Rejected       0.93      0.97      0.95      2235

    accuracy                           0.91      2413
   macro avg       0.60      0.56      0.57      2413
weighted avg       0.88      0.91      0.89      2413



## Conclusions

The resulting feature importance shows how the new "GoodFeature" and its correlated variant are on top of the list of features, and the "BadFeature" is at the bottom. You should consider the new features with high feature importance for inclusion in Pega.

We also do a correlation check, so very similar features are labeled with their group index. In this example, you would want only the best feature of "Group 1" to be included, the other one won't add much values since it is so highly correlated.
